# HRP Database Setup

This notebook does the following:
1. Loading health monitoring data from Excel files
2. Setting up a local SQLite database for the data storage
3. Creating a normalized schema out of measurements, seniors, and other tables.

**Summary of Results:**
- **Seniors**: 14,170 unique seniors
- **Measurements**: 164,331,155 measurements for different types
- **Medical Info**: 8,739 seniors with disease & medication data
- **Diseases**: 161 unique diseases
- **Medications**: 1,731 unique medications
- **SOS Alerts**: 8,983 alert records
- **Storage**: Local SQLite database

In [ ]:
import sys
import os
import time
import sqlite3
from pathlib import Path
import warnings

import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.components.load_data import load_all_data

warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Section 1: Load and Inspect Raw Excel Data

Load the new collection: three measurement files, medical/diseases, seniors demographics (gender, birthdate, age), and SOS alerts. Inspect structure, types, and basic quality.

In [63]:
# Define paths
raw_data_dir = Path("../data/raw/2026-02-08")
measurement_files = [
    raw_data_dir / "data-2025-12-01-07_202602091218.xlsx",
    raw_data_dir / "data-2025-12-08-14_202602091252.xlsx",
    raw_data_dir / "data-2025-12-15-21_202602091330.xlsx",
    raw_data_dir / "data-2025-12-22-28_202602091430.xlsx",
    raw_data_dir / "data-2025-12-29-2026-01-04_202602091614.xlsx",
    raw_data_dir / "data-2026-01-05-11-_202602092056.xlsx",
    raw_data_dir / "data-2026-01-12-18_202602100141.xlsx",
    raw_data_dir / "data-2026-01-19-25_202602100226.xlsx",
    raw_data_dir / "data-2026-01-26-31-.xlsx",
]
med_file = raw_data_dir / "Med&Diseases_202602082215.xlsx"
demo_file = raw_data_dir / "SeniorGenderAge_202602082214.xlsx"
sos_file = raw_data_dir / "SOS_202602082216.xlsx"

# Ensure all files exist
for p in measurement_files + [med_file, demo_file, sos_file]:
    assert p.exists(), f"Missing file: {p}"

In [64]:
# Load Medications & Disease Data
df_medical = pd.read_excel(med_file, engine="openpyxl")
df_medical.shape

(8743, 3)

In [65]:
df_medical.columns

Index(['seniorID', 'diseaseNames', 'medicineNames'], dtype='object')

In [66]:
df_medical.dtypes

seniorID          int64
diseaseNames     object
medicineNames    object
dtype: object

In [67]:
df_medical.head()

,seniorID,diseaseNames,medicineNames
0,2875,"Osteoporoza,Nadciśnienie tętnicze,Arytmia serc...","Acard,Emanera,Agen,Concor,Valzek"
1,3755,"Miażdzyca,Osteoporoza","Gensulin,Beto,Furosemidum,Amlopin,Zahron,Berod..."
2,3762,"Cukrzyca,Niedoczynnośc tarczycy,Niedoczynnośc ...","Letrox,Diosminex,Valsacor,Metformax,Bibloc,Pol..."
3,3805,"Stomia,Niedosłuch,Skolioza","Pregabalin,Staveran,Neurovit"
4,4367,"Miażdżyca kończyn dolnych,Niewydolnośc układu ...","Allupol,Cipropol,Eliquis,Ezehron,Areplex"


In [68]:
df_medical.isnull().sum()

seniorID         0
diseaseNames     0
medicineNames    0
dtype: int64

In [69]:
# Load Demographics Data
df_demo_raw = pd.read_excel(demo_file, engine="openpyxl")
df_demo_raw.shape

(14170, 4)

In [70]:
df_demo_raw.columns

Index(['seniorID', 'gender', 'birthDate', 'age'], dtype='object')

In [71]:
df_demo_raw.dtypes

seniorID       int64
gender        object
birthDate     object
age          float64
dtype: object

In [72]:
df_demo_raw.isnull().sum()

seniorID       0
gender         0
birthDate    309
age          309
dtype: int64

In [73]:
# Load SOS Alerts
df_sos = pd.read_excel(sos_file, engine="openpyxl")
df_sos.shape

(8983, 3)

In [74]:
df_sos.columns

Index(['seniorID', 'alertDate', 'sosNote'], dtype='object')

In [75]:
df_sos.dtypes

seniorID              int64
alertDate    datetime64[ns]
sosNote              object
dtype: object

In [76]:
df_sos.head()

,seniorID,alertDate,sosNote
0,3176,2026-01-07 19:33:57,Alarm. Nawiązano kontakt z Seniorem. Opiekun p...
1,3176,2025-12-10 19:30:51,Alarm przypadkowy
2,3176,2025-12-04 13:19:44,Alarm przypadkowy
3,3179,2025-12-19 13:43:28,Alarm testowy
4,3188,2026-01-04 06:12:46,Alarm przypadkowy


In [77]:
df_sos.isnull().sum()

seniorID      0
alertDate     0
sosNote      42
dtype: int64

In [78]:
# Measurement data sheets (inspect first file)
xls = pd.ExcelFile(measurement_files[0])
sheet_names = xls.sheet_names
print(f"Total sheets in first file: {len(sheet_names)}")
print(f"Sheet names: {sheet_names}\n")

Total sheets in first file: 21
Sheet names: ['expdata', 'expdata#1', 'expdata#2', 'expdata#3', 'expdata#4', 'expdata#5', 'expdata#6', 'expdata#7', 'expdata#8', 'expdata#9', 'expdata#10', 'expdata#11', 'expdata#12', 'expdata#13', 'expdata#14', 'expdata#15', 'expdata#16', 'expdata#17', 'expdata#18', 'expdata#19', 'expdata#20']



In [79]:
for i, sheet_name in enumerate(sheet_names[:2]):
    df = pd.read_excel(measurement_files[0], sheet_name=sheet_name, nrows=5, engine="openpyxl")
    print(f"\nSheet '{sheet_name}':")
    print(f"    Col 0 (seniorID): {df.iloc[:, 0].values[:2]}")
    print(f"    Col 1 (value): {df.iloc[:, 1].values[:2]}")
    print(f"    Col 2 (sbp): {df.iloc[:, 2].values[:2]}")
    print(f"    Col 3 (dbp): {df.iloc[:, 3].values[:2]}")
    print(f"    Col 4 (date): {df.iloc[:, 4].values[:2]}")
    print(f"    Col 5 (type): {df.iloc[:, 5].values[:2]}")


Sheet 'expdata':
    Col 0 (seniorID): [49789 45846]
    Col 1 (value): [36.6 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-12-04T18:00:11.000000000' '2025-12-04T18:00:11.000000000']
    Col 5 (type): ['Temperature' 'Temperature']

Sheet 'expdata#1':
    Col 0 (seniorID): [48684 49838]
    Col 1 (value): [36.6 36.4]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-12-06T02:34:11.000000000' '2025-12-06T02:34:12.000000000']
    Col 5 (type): ['Temperature' 'Temperature']


## Section 2: Store Data in SQLite Database

Use the pipeline from `src.components.load_data` to load demographics, measurements, medical info, and alerts into SQLite.

In [80]:
# Run end-to-end load using the following pipeline
# Use streaming to avoid large memory usage and commit in batches
# Set fresh_start=True to rebuild the DB and process all sheets from scratch
load_all_data(data_dir=raw_data_dir, fresh_start=True, streaming=True, batch_rows=100_000, resume=True)

INFO:src.components.load_data:Deleted existing database
INFO:src.components.database:Database initialized at c:\Users\eldar\Projects\AI-CVD\db\hrp_data.db
INFO:src.components.load_data:Loading seniors demographics from ..\data\raw\2026-02-08\SeniorGenderAge_202602082214.xlsx


INFO:src.components.load_data:Loaded 14170 senior demographic rows
INFO:src.components.load_data:Upserted 14170 seniors with demographics
INFO:src.components.load_data:Streaming measurements from ..\data\raw\2026-02-08\data-2025-12-01-07_202602091218.xlsx
INFO:src.components.load_data:  expdata: +100,000 (total 100,000)
INFO:src.components.load_data:  expdata: +100,000 (total 200,000)
INFO:src.components.load_data:  expdata: +100,000 (total 300,000)
INFO:src.components.load_data:  expdata: +100,000 (total 400,000)
INFO:src.components.load_data:  expdata: +100,000 (total 500,000)
INFO:src.components.load_data:  expdata: +100,000 (total 600,000)
INFO:src.components.load_data:  expdata: +100,000 (total 700,000)
INFO:src.components.load_data:  expdata: +100,000 (total 800,000)
INFO:src.components.load_data:  expdata: +100,000 (total 900,000)
INFO:src.components.load_data:  expdata: +100,000 (total 1,000,000)
INFO:src.components.load_data:  expdata: +48,575 (total 1,048,575)
INFO:src.compon


DATABASE SUMMARY
seniors.......................          14,170
measurements..................     164,331,155
alerts........................           8,983
medical_info (raw)............           8,739
diseases......................             161
medicines.....................           1,731
senior_diseases...............          52,917
senior_medicines..............          51,595


In [81]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

In [82]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [83]:
print(f"Database initialized at {db_path.as_posix()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

Database initialized at ../db/hrp_data.db
Database size: 32595344.0 KB


In [84]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables created: {[t[0] for t in tables]}")


Tables created: ['seniors', 'measurements', 'sqlite_sequence', 'medical_info', 'diseases', 'medicines', 'senior_diseases', 'senior_medicines', 'alerts', 'ingestion_state']


# Section 3: Verify/Inspect Data Integrity

In [85]:
# Pull measurements 
df_measurements = pd.read_sql("SELECT * FROM measurements LIMIT 100000", conn)
print(df_measurements.shape)
df_measurements.head()

(100000, 7)


,id,senior_id,value,sbp,dbp,date,type
0,1,49789,36.6,None,None,2025-12-04 18:00:11,Temperature
1,2,45846,36.6,None,None,2025-12-04 18:00:11,Temperature
2,3,21927,36.5,None,None,2025-12-04 18:00:11,Temperature
3,4,38074,36.6,None,None,2025-12-04 18:00:11,Temperature
4,5,49987,36.7,None,None,2025-12-04 18:00:11,Temperature


In [86]:
# Medical information counts
counts_med = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM medical_info", conn)
counts_med

,n_rows,n_seniors
0,8739,8739


In [87]:
# Alerts counts
counts_alerts = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM alerts", conn)
counts_alerts

,n_rows,n_seniors
0,8983,4382


In [88]:
# Preview alerts
pd.read_sql("SELECT * FROM alerts LIMIT 5", conn)

,alert_id,senior_id,alert_date,sos_note
0,1,3176,2026-01-07 19:33:57,Alarm. Nawiązano kontakt z Seniorem. Opiekun p...
1,2,3176,2025-12-10 19:30:51,Alarm przypadkowy
2,3,3176,2025-12-04 13:19:44,Alarm przypadkowy
3,4,3179,2025-12-19 13:43:28,Alarm testowy
4,5,3188,2026-01-04 06:12:46,Alarm przypadkowy


In [89]:
# Measurement type distribution
measure_type_counts = pd.read_sql(
    "SELECT type, COUNT(*) AS cnt FROM measurements GROUP BY type ORDER BY cnt DESC",
    conn,
)
measure_type_counts

,type,cnt
0,Heartrate,37260275
1,BloodPressure,37260213
2,Temperature,37117542
3,Saturation,29911518
4,Steps,22781607


## Section 4: Query and Validate Stored Data

Execute SQL queries to retrieve data and perform basic analysis to confirm database functionality.

### Example 1: Get all measurements for a specific type

In [90]:
query1 = """
    SELECT senior_id, value, date, type
    FROM measurements
    WHERE type = 'Heartrate'
    ORDER BY date
    LIMIT 10
"""

In [91]:
df_example1 = pd.read_sql(query1, conn)
df_example1

,senior_id,value,date,type
0,38873,100.0,2025-12-01 00:00:10,Heartrate
1,43678,62.0,2025-12-01 00:00:10,Heartrate
2,33485,100.0,2025-12-01 00:00:10,Heartrate
3,9086,49.0,2025-12-01 00:00:10,Heartrate
4,46563,58.0,2025-12-01 00:00:10,Heartrate
5,52344,67.0,2025-12-01 00:00:10,Heartrate
6,33280,56.0,2025-12-01 00:00:10,Heartrate
7,9080,63.0,2025-12-01 00:00:10,Heartrate
8,46663,48.0,2025-12-01 00:00:10,Heartrate
9,46347,80.0,2025-12-01 00:00:10,Heartrate


### Example 2: Aggregate statistics by measurement type

In [92]:
query2 = """
    SELECT 
        type,
        COUNT(*) as measurement_count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        AVG(value) as avg_value,
        MIN(value) as min_value,
        MAX(value) as max_value,
        ROUND(AVG(value), 2) as mean
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY measurement_count DESC
"""

In [93]:
df_example2 = pd.read_sql(query2, conn)
df_example2

,type,measurement_count,unique_seniors,avg_value,min_value,max_value,mean
0,Heartrate,37260275,14011,73.253230,1.0,214.0,73.25
1,Temperature,37117542,13990,36.664781,36.3,127.9,36.66
2,Saturation,29911518,13960,97.049382,80.0,99.0,97.05
3,Steps,22781607,13300,2901.511171,1.0,48854.0,2901.51


### Example 3: Get measurements for a specific senior

In [94]:
sample_senior_id = int(df_measurements.iloc[0]["senior_id"])
query3 = """
    SELECT senior_id, value, sbp, dbp, date, type
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date DESC
    LIMIT 10
"""

In [95]:
df_example3 = pd.read_sql(query3, conn, params=[sample_senior_id])
df_example3

,senior_id,value,sbp,dbp,date,type
0,49789,99.0,NaN,NaN,2026-01-13 11:25:11,Saturation
1,49789,NaN,131.0,82.0,2026-01-13 11:25:11,BloodPressure
2,49789,113.0,NaN,NaN,2026-01-13 11:25:11,Heartrate
3,49789,36.6,NaN,NaN,2026-01-13 11:25:11,Temperature
4,49789,278.0,NaN,NaN,2026-01-13 11:23:01,Steps
5,49789,99.0,NaN,NaN,2026-01-13 11:15:11,Saturation
6,49789,NaN,142.0,83.0,2026-01-13 11:15:11,BloodPressure
7,49789,122.0,NaN,NaN,2026-01-13 11:15:11,Heartrate
8,49789,36.6,NaN,NaN,2026-01-13 11:15:11,Temperature
9,49789,99.0,NaN,NaN,2026-01-13 11:05:11,Saturation


### Example 4: Query performance test

In [96]:
start = time.time()
query4 = "SELECT * FROM measurements WHERE type = 'Heartrate' LIMIT 1000"
df_example4 = pd.read_sql(query4, conn)
elapsed = time.time() - start

In [97]:
print(f"  Retrieved {len(df_example4)} rows in {elapsed:.4f} seconds")

  Retrieved 1000 rows in 0.0051 seconds


### Example 4: Blood Pressure Analysis

In [98]:
query5 = """
    SELECT senior_id, sbp, dbp, date, type
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
    ORDER BY date DESC
    LIMIT 10
"""

In [99]:
df_example5 = pd.read_sql(query5, conn)
df_example5

,senior_id,sbp,dbp,date,type
0,22282,135.0,64.0,2026-01-31 23:59:30,BloodPressure
1,13258,137.0,73.0,2026-01-31 23:59:30,BloodPressure
2,38877,125.0,70.0,2026-01-31 23:59:25,BloodPressure
3,53969,122.0,76.0,2026-01-31 23:59:23,BloodPressure
4,38965,112.0,74.0,2026-01-31 23:59:23,BloodPressure
5,53795,132.0,87.0,2026-01-31 23:59:23,BloodPressure
6,43762,146.0,81.0,2026-01-31 23:59:23,BloodPressure
7,48837,136.0,76.0,2026-01-31 23:59:23,BloodPressure
8,16457,135.0,76.0,2026-01-31 23:59:23,BloodPressure
9,53683,129.0,75.0,2026-01-31 23:59:23,BloodPressure


### Example 5: Get Blood Pressure Statistics

In [100]:
query5_stats = """
    SELECT 
        COUNT(*) as bp_measurements,
        COUNT(DISTINCT senior_id) as seniors_with_bp,
        AVG(sbp) as avg_systolic,
        AVG(dbp) as avg_diastolic,
        MIN(sbp) as min_systolic,
        MAX(sbp) as max_systolic,
        MIN(dbp) as min_diastolic,
        MAX(dbp) as max_diastolic
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
"""

In [101]:
df_bp_stats = pd.read_sql(query5_stats, conn)
df_bp_stats

,bp_measurements,seniors_with_bp,avg_systolic,avg_diastolic,min_systolic,max_systolic,min_diastolic,max_diastolic
0,37260213,14011,129.432691,78.566553,68.0,212.0,23.0,157.0
